# ⚔️ Support Vector Machines (SVM) — Mencari Pemisah Terbaik

**Modul 1: Machine Learning Fundamentals** | Notebook 5 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Apa itu **SVM** dan konsep **margin** serta **support vectors**
2. Perbedaan **kernel** (linear, RBF, polynomial) dan kapan menggunakannya
3. Perbandingan SVM vs model-model sebelumnya (Decision Tree, KNN)
4. Aplikasi SVM pada dataset yang lebih besar *(bonus)*

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Analogi Sederhana

Bayangkan ada dua kelompok siswa di lapangan — **tim merah** dan **tim biru**. Kamu diminta menarik **garis pemisah** di antara mereka.

Banyak garis yang bisa memisahkan, tapi mana yang **terbaik**?

**SVM** memilih garis yang memiliki **jarak (margin) terlebar** dari kedua kelompok. Mengapa? Karena garis dengan margin lebar lebih **stabil** — kalau ada siswa baru, kita lebih yakin kelompoknya.

```
     Tim Merah          Garis Pemisah          Tim Biru
  🔴  🔴               |                      🔵  🔵
    🔴    🔴    ←margin→|←margin→       🔵   🔵
  🔴        🔴          |                    🔵
      🔴  🔴            |               🔵    🔵
```

**Support Vectors** = titik data yang paling dekat dengan garis pemisah. Mereka yang "mendukung" posisi garis.

---

## 1️⃣ Persiapan

In [ ]:
# Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Library berhasil di-import!')

---

## 2️⃣ Memuat Data

Kita kembali menggunakan **Social Network Ads** — dataset yang sama dengan notebook Decision Tree dan KNN. Dengan begitu, kita bisa **membandingkan performa** ketiga model di akhir!

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
data = pd.read_csv('Social_Network_Ads.csv')
print(f'Dataset: {data.shape[0]} baris, {data.shape[1]} kolom')
data.head()

In [ ]:
# Persiapan data (sama seperti notebook sebelumnya)
X = data.drop(columns='Purchased')
y = data['Purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Data latihan: {X_train.shape[0]} | Data ujian: {X_test.shape[0]}')

---

## 3️⃣ SVM dengan Kernel Linear

Kernel **linear** membuat garis lurus untuk memisahkan dua kelas. Ini yang paling sederhana.

### Apa itu Margin dan Support Vectors?

Mari kita visualisasikan konsep ini secara langsung.

In [ ]:
# Latih SVM dengan kernel linear
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_scaled, y_train)

# Evaluasi
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)
f1_linear = f1_score(y_test, y_pred_linear)

print(f'📊 SVM Linear')
print(f'  Accuracy: {acc_linear:.3f} | F1-Score: {f1_linear:.3f}')
print(f'  Jumlah Support Vectors: {svm_linear.n_support_}')
print(f'  (Kelas 0: {svm_linear.n_support_[0]}, Kelas 1: {svm_linear.n_support_[1]})')

In [ ]:
# Visualisasi Decision Boundary + Support Vectors
x_min, x_max = X_train_scaled[:, 0].min() - 0.5, X_train_scaled[:, 0].max() + 0.5
y_min, y_max = X_train_scaled[:, 1].min() - 0.5, X_train_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                      np.arange(y_min, y_max, 0.02))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Decision Boundary
Z = svm_linear.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
axes[0].scatter(X_train_scaled[:, 0], X_train_scaled[:, 1],
                c=y_train, cmap='RdYlGn', edgecolors='white', s=40, alpha=0.6)
axes[0].set_title('Decision Boundary — SVM Linear', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Umur (scaled)', fontsize=11)
axes[0].set_ylabel('Gaji (scaled)', fontsize=11)

# Plot 2: Highlight Support Vectors
axes[1].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
axes[1].scatter(X_train_scaled[:, 0], X_train_scaled[:, 1],
                c=y_train, cmap='RdYlGn', edgecolors='white', s=30, alpha=0.3)

# Tandai support vectors dengan lingkaran besar
sv = svm_linear.support_vectors_
axes[1].scatter(sv[:, 0], sv[:, 1], s=120, facecolors='none',
                edgecolors='blue', linewidths=2, label='Support Vectors')
axes[1].set_title('Support Vectors (lingkaran biru)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Umur (scaled)', fontsize=11)
axes[1].set_ylabel('Gaji (scaled)', fontsize=11)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print('💡 Support Vectors = titik-titik yang PALING DEKAT dengan garis pemisah.')
print('   Mereka menentukan posisi dan orientasi garis. Titik lain tidak berpengaruh!')

---

## 4️⃣ Kernel Trick — Ketika Garis Lurus Tidak Cukup

Kadang data tidak bisa dipisahkan dengan garis lurus. Contoh: bayangkan titik merah di **tengah** dan titik biru **mengelilinginya** — tidak ada garis lurus yang bisa memisahkan!

**Kernel trick** menyelesaikan ini dengan cara "mengangkat" data ke dimensi yang lebih tinggi, di mana pemisahan menjadi mungkin.

### Jenis-jenis Kernel:

| Kernel | Cara Kerja | Cocok Untuk |
|--------|-----------|-------------|
| **Linear** | Garis lurus | Data yang sudah terpisah jelas |
| **RBF (Radial Basis Function)** | Batas melengkung | Kebanyakan kasus (default) |
| **Polynomial** | Batas bentuk polinomial | Hubungan non-linear tertentu |

Mari kita bandingkan ketiga kernel!

In [ ]:
# Bandingkan 3 kernel
kernels = [
    ('Linear', 'linear'),
    ('RBF (Radial)', 'rbf'),
    ('Polynomial (degree=3)', 'poly')
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
hasil_kernel = {}

for idx, (nama, kernel) in enumerate(kernels):
    svm_temp = SVC(kernel=kernel, random_state=42)
    svm_temp.fit(X_train_scaled, y_train)

    y_pred_temp = svm_temp.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_temp)
    f1 = f1_score(y_test, y_pred_temp)
    hasil_kernel[nama] = {'Accuracy': acc, 'F1': f1, 'SV': sum(svm_temp.n_support_)}

    Z = svm_temp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    axes[idx].scatter(X_test_scaled[:, 0], X_test_scaled[:, 1],
                      c=y_test, cmap='RdYlGn', edgecolors='white', s=50, alpha=0.8)

    axes[idx].set_title(f'{nama}\nAcc={acc:.3f} | F1={f1:.3f}',
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Umur (scaled)', fontsize=10)
    axes[idx].set_ylabel('Gaji (scaled)', fontsize=10)

plt.suptitle('Perbandingan Kernel SVM', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Tabel perbandingan
print('📊 Perbandingan Kernel SVM')
print('=' * 55)
print(f'{"Kernel":<25} {"Accuracy":>10} {"F1-Score":>10} {"Support V.":>10}')
print('-' * 55)
for nama, metrics in hasil_kernel.items():
    print(f'  {nama:<23} {metrics["Accuracy"]:>10.3f} {metrics["F1"]:>10.3f} {metrics["SV"]:>10}')
print('=' * 55)

print('\n💡 RBF biasanya paling bagus karena batasnya bisa melengkung.')
print('   Linear paling cepat tapi batasnya kaku (garis lurus).')

---

## 5️⃣ Parameter C — Mengontrol Keseimbangan

Parameter **C** mengontrol trade-off antara:
- **C kecil** → Margin lebar, tapi boleh ada beberapa kesalahan (lebih toleran)
- **C besar** → Margin sempit, berusaha benar semua (lebih ketat, risiko overfitting)

**Analogi:** Seperti guru yang menilai ujian:
- C kecil = guru yang santai, boleh ada beberapa kesalahan selama secara umum paham
- C besar = guru yang ketat, setiap jawaban harus benar

In [ ]:
# Bandingkan berbagai nilai C dengan kernel RBF
c_values = [0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for idx, c_val in enumerate(c_values):
    svm_c = SVC(kernel='rbf', C=c_val, random_state=42)
    svm_c.fit(X_train_scaled, y_train)

    Z = svm_c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    axes[idx].scatter(X_test_scaled[:, 0], X_test_scaled[:, 1],
                      c=y_test, cmap='RdYlGn', edgecolors='white', s=30, alpha=0.8)

    acc_c = accuracy_score(y_test, svm_c.predict(X_test_scaled))
    axes[idx].set_title(f'C={c_val}\nAcc={acc_c:.3f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Umur', fontsize=9)
    if idx == 0:
        axes[idx].set_ylabel('Gaji', fontsize=9)

plt.suptitle('Pengaruh Parameter C pada SVM (Kernel RBF)', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print('💡 C kecil → Batas lebih halus (toleran), C besar → Batas lebih ketat (detail)')

---

## 6️⃣ Evaluasi Model Terbaik

In [ ]:
# Gunakan SVM RBF (biasanya terbaik)
svm_best = SVC(kernel='rbf', C=1.0, random_state=42)
svm_best.fit(X_train_scaled, y_train)
y_pred_best = svm_best.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['Tidak Beli', 'Beli']).plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — SVM RBF', fontsize=13, fontweight='bold')
plt.show()

acc = accuracy_score(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)
print(f'\n📊 SVM RBF: Accuracy={acc:.3f}, F1-Score={f1:.3f}')

---

## 7️⃣ Perbandingan Besar: SVM vs Decision Tree vs KNN

Sekarang kita bandingkan **semua model** yang sudah kita pelajari pada dataset yang sama!

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Latih semua model
models = {
    'Decision Tree (depth=4)': DecisionTreeClassifier(max_depth=4, random_state=0),
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM Linear': SVC(kernel='linear', random_state=42),
    'SVM RBF': SVC(kernel='rbf', random_state=42),
}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

print('📊 Perbandingan Semua Model')
print('=' * 55)
print(f'{"Model":<25} {"Accuracy":>10} {"F1-Score":>10}')
print('-' * 55)

for idx, (nama, mdl) in enumerate(models.items()):
    mdl.fit(X_train_scaled, y_train)
    y_pred_m = mdl.predict(X_test_scaled)

    acc_m = accuracy_score(y_test, y_pred_m)
    f1_m = f1_score(y_test, y_pred_m)
    print(f'  {nama:<23} {acc_m:>10.3f} {f1_m:>10.3f}')

    Z = mdl.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    axes[idx].scatter(X_test_scaled[:, 0], X_test_scaled[:, 1],
                      c=y_test, cmap='RdYlGn', edgecolors='white', s=40, alpha=0.8)
    axes[idx].set_title(f'{nama}\nAcc={acc_m:.3f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Umur (scaled)', fontsize=9)
    if idx == 0:
        axes[idx].set_ylabel('Gaji (scaled)', fontsize=9)

print('=' * 55)

plt.suptitle('Perbandingan Decision Boundary — Semua Model', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

**Pengamatan:**
- **Decision Tree** → Batas kotak-kotak (garis horizontal/vertikal)
- **KNN** → Batas berliku mengikuti distribusi data
- **SVM Linear** → Satu garis lurus memisahkan dua kelompok
- **SVM RBF** → Batas melengkung halus — biasanya paling akurat

---

## 🌟 BONUS: SVM pada Dataset Lebih Besar

Sekarang kita coba SVM pada dataset **Income Evaluation** — prediksi apakah seseorang berpenghasilan **di atas atau di bawah $50K per tahun** berdasarkan data demografis.

Dataset ini jauh lebih besar (32K+ baris) dan kompleks (14 fitur).

In [ ]:
# Upload income_evaluation.csv
uploaded2 = files.upload()

In [ ]:
# Baca dan bersihkan dataset
df = pd.read_csv('income_evaluation.csv')

# Bersihkan nama kolom (ada spasi di depan)
df.columns = [col.strip() for col in df.columns]

# Bersihkan nilai string
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print(f'Dataset: {df.shape[0]:,} baris, {df.shape[1]} kolom')
print(f'\nTarget (income):')
print(df['income'].value_counts())

# Buat target numerik
df['target'] = (df['income'] == '>50K').astype(int)

# Pilih fitur numerik saja (untuk kesederhanaan)
fitur_numerik = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
X_inc = df[fitur_numerik]
y_inc = df['target']

print(f'\nFitur yang digunakan: {fitur_numerik}')

In [ ]:
# Split dan scale
X_train_inc, X_test_inc, y_train_inc, y_test_inc = train_test_split(
    X_inc, y_inc, test_size=0.2, random_state=42
)

scaler_inc = StandardScaler()
X_train_inc_s = scaler_inc.fit_transform(X_train_inc)
X_test_inc_s = scaler_inc.transform(X_test_inc)

# Latih SVM RBF
svm_income = SVC(kernel='rbf', C=1.0, random_state=42)
svm_income.fit(X_train_inc_s, y_train_inc)

y_pred_inc = svm_income.predict(X_test_inc_s)
acc_inc = accuracy_score(y_test_inc, y_pred_inc)
f1_inc = f1_score(y_test_inc, y_pred_inc)

print(f'📊 SVM RBF pada Dataset Income')
print(f'  Data training: {len(X_train_inc):,} | Data test: {len(X_test_inc):,}')
print(f'  Accuracy: {acc_inc:.3f} ({acc_inc*100:.1f}%)')
print(f'  F1-Score: {f1_inc:.3f} ({f1_inc*100:.1f}%)')

# Confusion Matrix
cm_inc = confusion_matrix(y_test_inc, y_pred_inc)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_inc, display_labels=['≤50K', '>50K']).plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Income Prediction', fontsize=13, fontweight='bold')
plt.show()

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **SVM** mencari garis/bidang pemisah dengan **margin terlebar** antara dua kelas
2. **Support Vectors** = titik data terdekat dengan garis pemisah — hanya mereka yang menentukan posisi garis
3. **Kernel trick** memungkinkan SVM memisahkan data yang tidak bisa dipisah dengan garis lurus:
   - Linear → garis lurus
   - RBF → melengkung (default, biasanya terbaik)
   - Polynomial → kurva polinomial
4. **Parameter C** mengontrol keseimbangan antara margin lebar dan ketepatan

### Perbandingan Model yang Sudah Dipelajari

| Model | Batas Keputusan | Butuh Scaling? | Kecepatan | Interpretasi |
|-------|----------------|---------------|-----------|-------------|
| Decision Tree | Kotak-kotak | Tidak | Cepat | Mudah |
| KNN | Berliku-liku | Ya | Lambat (data besar) | Sulit |
| SVM Linear | Garis lurus | Ya | Sedang | Sedang |
| SVM RBF | Melengkung halus | Ya | Lambat (data besar) | Sulit |

### Kapan Menggunakan SVM?

✅ Dataset **kecil-menengah** dengan fitur yang banyak
✅ Kamu butuh model dengan **akurasi tinggi**
✅ Data memiliki **margin yang jelas** antara kelas

❌ Dataset **sangat besar** (lambat untuk training)
❌ Kamu butuh model yang **mudah dijelaskan** ke orang awam

---

### 🔜 Notebook Selanjutnya: Ensemble Learning
Bagaimana jika kita **menggabungkan banyak model** menjadi satu tim? Itulah ide di balik Ensemble Methods — Voting, Bagging, Boosting, dan Stacking!